# SETTING

In [1]:
from pynq import allocate
from pynq import Overlay
import numpy as np

overlay = Overlay('gempy.bit')

/usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/ps.py:434: UserWarning: Setting frequency to the closest possible value 98.8879MHz.
  warnings.warn(


In [2]:
from pynq import Clocks
Clocks.fclk0_mhz = 200
print(f'CPU:   {Clocks.cpu_mhz:.6f}MHz')
print(f'FCLK0: {Clocks.fclk0_mhz:.6f}MHz')

CPU:   1199.988000MHz
FCLK0: 211.902643MHz


/usr/local/share/pynq-venv/lib/python3.10/site-packages/pynq/ps.py:434: UserWarning: Setting frequency to the closest possible value 211.90264MHz.
  warnings.warn(


In [3]:
def alu(data):
    # 分配 DMA 緩衝區
    in_buffer  = allocate(shape=(np.size(data),), dtype=np.int64)
    out_buffer = allocate(shape=(data[0]+5,), dtype=np.int16)
    
    # 複製資料並同步
    np.copyto(in_buffer, data)
    in_buffer.sync_to_device()
    
    # 觸發 DMA
    overlay.axi_dma_0.recvchannel.transfer(out_buffer)
    overlay.axi_dma_0.sendchannel.transfer(in_buffer)
    overlay.axi_dma_0.sendchannel.wait()
    overlay.axi_dma_0.recvchannel.wait()
    
    out_buffer.sync_from_device()
    
    # 釋放 in_buffer
    in_buffer.freebuffer()
    
    return out_buffer

# SERVER

In [ ]:
import socket
import struct

IN_DT = np.int64
IN_BYTES_PER = 8
SOCK_BUF_SIZE = 8 * 1024 * 1024  # 8MB Buffer


with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    s.setsockopt(socket.SOL_SOCKET, socket.SO_RCVBUF, SOCK_BUF_SIZE)
    s.setsockopt(socket.SOL_SOCKET, socket.SO_SNDBUF, SOCK_BUF_SIZE)
    
    s.bind(("0.0.0.0", 8765))
    s.listen(1)
    print("PYNQ Server listening on port 8765 ...")

    header_buf = bytearray(8)
    header_view = memoryview(header_buf)

    # 準備一個單純用來接網路資料的 NumPy 陣列 (不耗費 CMA 資源)
    cap_elems = 1024
    net_inbuf = np.empty(cap_elems, dtype=IN_DT)

    while True:
        conn, addr = s.accept()
        print(f"Connected by {addr}")
        
        with conn:
            conn.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
            
            while True:
                try:
                    # 1. 讀取資料總長度 (8 bytes)
                    if conn.recv_into(header_view, 8, socket.MSG_WAITALL) < 8:
                        break  # Client 斷線
                    
                    nbytes = struct.unpack("<Q", header_view)[0]
                    elems = nbytes // IN_BYTES_PER
                    
                    # 2. 動態調整網路接收緩衝區大小
                    if elems > cap_elems:
                        net_inbuf = np.empty(elems, dtype=IN_DT)
                        cap_elems = elems

                    # 3. 接收 Client 傳來的陣列資料
                    target_view = net_inbuf.view(np.uint8)[:nbytes]
                    conn.recv_into(target_view, nbytes, socket.MSG_WAITALL)

                    # ======================================
                    # 4. 呼叫 ALU 送入硬體計算
                    # ======================================
                    # 傳入剛剛收到的陣列切片
                    out = alu(net_inbuf[:elems])

                    # 5. 回傳計算結果給 Client
                    # 打包回傳長度 (out.nbytes = 元素數量 * 2 bytes)
                    struct.pack_into("<Q", header_buf, 0, out.nbytes)
                    conn.sendall(header_buf) 
                    
                    # 將 DMA 算完的結果直接丟進網路送出
                    conn.sendall(out)
                    
                    # 因為 out_buffer 是在 alu 裡 allocate 的，
                    # 網路傳送完畢後，必須手動把它 free 掉！
                    out.freebuffer()
                    
                except Exception as e:
                    print(f"Connection error: {e}")
                    break
        print(f"Disconnected from {addr}")

PYNQ Server listening on port 8765 ...
Connected by ('192.168.3.139', 3142)
Disconnected from ('192.168.3.139', 3142)
Connected by ('192.168.3.139', 3104)
Disconnected from ('192.168.3.139', 3104)
Connected by ('192.168.3.139', 3140)
Disconnected from ('192.168.3.139', 3140)
Connected by ('192.168.3.139', 3106)
